In [2]:
import pandas as pd
import plotly.express as px
from trains import train_counts

# 1. Load train data
df_trains = train_counts().reset_index()
df_trains = df_trains.rename(columns={"index": "timestamp"})

# 2. Get columns for sensor GASA-01-C
meta = pd.read_csv("sensor_mapping_with_coords.csv")
cols = meta[meta["sensor_id"] == "GASA-01-C"]["column_name"].tolist()

# 3. Load pedestrian flow selecting specific columns
columns_to_read = ["timestamp"] + cols
df_flow = pd.read_csv(
    "SAIL2025_LVMA_data_3min_20August-25August2025_flow.csv",
    usecols=columns_to_read,
)

# Convert timestamp and sum pedestrian counts for the sensor
df_flow["timestamp"] = pd.to_datetime(df_flow["timestamp"])
df_flow["pedestrians"] = df_flow[cols].sum(axis=1)

# 4. Merge data and filter for August 20
df = pd.merge(df_trains, df_flow, on="timestamp")
df = df[df["timestamp"].dt.strftime("%Y-%m-%d") == "2025-08-20"]

# 5. Plotly Express Chart
fig = px.bar(
    df,
    x="timestamp",
    y="arrivals",
    title="Train Arrivals vs Pedestrian Flow",
    color_discrete_sequence=["orange"],
)

fig_line = px.line(
    df, x="timestamp", y="pedestrians", color_discrete_sequence=["blue"]
)
fig_line.update_traces(yaxis="y2")
fig.add_traces(fig_line.data)

fig.update_layout(
    yaxis_title="Train Arrivals",
    yaxis2=dict(title="Pedestrian Count", overlaying="y", side="right"),
)

fig.show()